In [2]:
import torch 
import torch.nn as nn 
import torch.optim as optim 

import torchvision  # Official pytorch lib. for Computer Vision
from torchvision.datasets import CIFAR10

In [3]:
# Datasets and dataloaders 
from torch.utils.data import DataLoader 
import torchvision.transforms as transforms # Gives utilities to transform an image

transform= transforms.Compose([ # Compose lets ous prepare a list of transformations which will be later applied on each image.
    transforms.ToTensor(), # Here pytorch will convert all data into tensor along with automatic scaling.
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # these are the predefined value we set to get a perticular std.deviation and mean value for CIFAR
])

trainset= CIFAR10(root= "./data", train= True, download=False, transform=transform) # It will download data take the training part and perform "transform" operations
testset= CIFAR10(root= "./data", train= False, download=False, transform=transform)

In [6]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [8]:
trainloader= DataLoader(trainset, batch_size=64, shuffle= True)
testloader= DataLoader(testset, batch_size=64)

# Building the cnn

In [20]:
class CNN(nn.Module): 
    # Constructor
    def __init__(self): 
        super(CNN, self).__init__()

        # convulation layer 
        self.conv_layers= nn.Sequential(
            # 1st layer
            nn.Conv2d(3, 32,kernel_size=3, padding= 1), # 3 is no. of i/p channel(R,G,B) and 32 is no. of o/p channels ,means convu. layer will learn 32 different features from an image. 
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # X=2 is kernel size and Y=2 is stride value

            # 2nd layer
            nn.Conv2d(32, 64,kernel_size=3, padding= 1),  
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
                
            # 3rd layer    
            nn.Conv2d(64, 128,kernel_size=3, padding= 1) , 
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        # Actual Ann Training 
        self.fc_layers= nn.Sequential(
            # 1st layer
            nn.Linear(4*4*128, 256), # Here its taking a linear(1D/tensor) layer with 4*4*128 i/p's and will give 256(random no.) o/p through 256 neurons. 
            nn.ReLU(),
                
            # 2nd layer
            nn.Linear(256, 10) 
        )

    # Forward pass
    def forward(self, x):
        x= self.conv_layers(x) 
        x= x.view(x.size(0), -1) # Flattening the o/p got from convu. layer and provided it to fully conn.(fc)
        x= self.fc_layers(x)

        return x # providing final o/p

In [22]:
model= CNN()

In [24]:
criterian =nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters())

# Training the CNN

In [31]:
epochs= 10

for epoch in range(epochs): 
    epoch_trainig_loss= 0.0

    for images, labels in trainloader: 
        optimizer.zero_grad()

        output = model.forward(images)  # Forward prorpo. 
        loss= criterian(output, labels) # loss function 
        loss.backward() # Backword propo. 

        optimizer.step() # updating parameter

        epoch_trainig_loss+= loss.item()

    print(f"epoch= {epoch} & loss= {epoch_trainig_loss/len(trainloader)}")

epoch= 0 & loss= 0.7631485558607999
epoch= 1 & loss= 0.6364189921056523
epoch= 2 & loss= 0.5386583912936623
epoch= 3 & loss= 0.43880307946897223
epoch= 4 & loss= 0.3627174035705569
epoch= 5 & loss= 0.2860115997474212
epoch= 6 & loss= 0.2272772688437682
epoch= 7 & loss= 0.1801617646880467
epoch= 8 & loss= 0.15001090742943005
epoch= 9 & loss= 0.11407458265323925


In [35]:
# Evaluate our CNN

correct_labels= 0 
total_labels= 0

model.eval()

with torch.no_grad():
    for images ,labels in testloader:
        output= model.forward(images)
        _, predicted= torch.max(output, 1)

        correct_labels+= (predicted == labels).sum().item()
        total_labels+= labels.size(0)

print(f"accuracy={correct_labels/total_labels*100}")

accuracy=74.74
